# mlx

> [mlx-lm](https://github.com/ml-explore/mlx-lm) on Apple Silicon: an explicit prompt cache, speculative decoding, quantized KV, and mlx-vlm routing for vision and audio.

Linux CI skips real-model cells with `skip_exec: true`. Shared `Chat` behaviour is in
[index](index.html) and [core](core.html).


In [ ]:
#| default_exp mlx

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, os
from base64 import b64decode
from tempfile import TemporaryDirectory
import mlx.core as mx
from mlx_lm import load as mlx_load, stream_generate
from mlx_lm.sample_utils import make_sampler
from mlx_lm.models.cache import (make_prompt_cache, trim_prompt_cache, can_trim_prompt_cache,
                                 save_prompt_cache, load_prompt_cache)
from huggingface_hub import hf_hub_download, scan_cache_dir
from fastcore.funccall import get_schema, mk_ns
from fastcore.all import Path, store_attr, patch, L, ifnone, first, listify
from rishi import core
from rishi.core import *

/Users/71293/code/personal/orgs/ramabana/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#| export
# re-exported so `from rishi.mlx import *` gives the same vocabulary as the other backends
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'mk_tr_details', 'truncated', 'hitl_policy', 'extract_fence',
         'mk_toolspec', 'split_think', 'parse_tool_tags', 'StreamSplit']

In [ ]:
from fastcore.test import test_eq, test_fail, test_close

## Loading models

MLX models are HuggingFace repos of quantized weights, most of them under
[mlx-community](https://huggingface.co/mlx-community). litert takes one `.litertlm` file and llama
takes one `.gguf` file, but an MLX model is a *directory*. `mlx_lm.load` takes the repo id straight
and handles the download and cache itself, so there is no file-picking ladder to write here.

`read_config` fetches just the repo's `config.json`. Two things are decided from it before loading:
the context window, and whether this is a vision or audio model that needs `mlx-vlm` rather than
plain `mlx-lm`.


In [ ]:
#| export
#: Common mlx-community repo ids.
qwen3_06b  = 'mlx-community/Qwen3-0.6B-4bit'    # tiny; useful as a speculative-decoding draft model
qwen3_17b  = 'mlx-community/Qwen3-1.7B-4bit'
DFLT_MAX_TOKENS = 1024   #: output cap when nothing says otherwise
qwen3_4b   = 'mlx-community/Qwen3-4B-4bit'
qwen3_8b   = 'mlx-community/Qwen3-8B-4bit'
qwen3_30b  = 'mlx-community/Qwen3-30B-A3B-4bit'
gemma3_4b  = 'mlx-community/gemma-3-4b-it-4bit'
llama32_3b = 'mlx-community/Llama-3.2-3B-Instruct-4bit'
qwen3vl_4b = 'mlx-community/Qwen3-VL-4B-Instruct-4bit'   # vision + text, needs `rishi[mlx-vlm]`
gemma4_e4b = 'mlx-community/gemma-4-e4b-it-4bit'         # vision + audio, needs `rishi[mlx-vlm]`
qwen3omni_30b = 'mlx-community/Qwen3-Omni-30B-A3B-Instruct-4bit'   # audio in, needs `rishi[mlx-vlm]`

def read_config(model):
    "A model's `config.json` as a dict, from a local directory or the hub, or `{}` if it cannot be read."
    try:
        if (p := Path(model)/'config.json').exists(): return json.loads(p.read_text())
    except OSError: pass
    try: return json.loads(Path(hf_hub_download(str(model), 'config.json')).read_text())
    except Exception: return {}

# a vision/audio model carries a second tower in its config; the key varies by architecture
_vlm_keys = ('vision_config', 'audio_config', 'vision_tower', 'image_token_index')

def is_vlm(model, cfg=None):
    "Does `model` declare a vision/audio tower, so it needs `mlx-vlm` rather than plain `mlx-lm`?"
    return any(k in (read_config(model) if cfg is None else cfg) for k in _vlm_keys)

def ctx_len(cfg, dflt=8192):
    "Context window from a model config (checking the nested text config VLMs use), else `dflt`."
    return cfg.get('max_position_embeddings') or (cfg.get('text_config') or {}).get('max_position_embeddings') or dflt

def cached_models():
    "MLX repo ids already in the local HuggingFace cache (no network)."
    try: return L(r.repo_id for r in scan_cache_dir().repos if 'mlx' in r.repo_id.lower())
    except Exception: return L()


In [ ]:
#| export
class MlxEngine:
    "A loaded MLX model and its tokenizer, plus an optional draft model for speculative decoding."
    def __init__(self, model, tokenizer, model_id=None, cfg=None, draft_model=None, draft_model_id=None):
        store_attr(); self.cfg = cfg or {}
    def __repr__(self): return f'MlxEngine({self.model_id})'
    def tokenize(self, text):
        "Token ids for `text`."
        return self.tokenizer.encode(text)
    @property
    def ctx_limit(self):
        "The model's context window, per its config."
        return ctx_len(self.cfg)
    def close(self):
        "Drop references to the weights so MLX can free them."
        self.model = self.draft_model = None

In [ ]:
test_eq(ctx_len({'max_position_embeddings': 40960}), 40960)
test_eq(ctx_len({'text_config': {'max_position_embeddings': 128000}}), 128000)   # VLMs nest it
test_eq(ctx_len({}), 8192)                                                       # fallback
assert is_vlm(None, cfg={'vision_config': {}})
assert is_vlm(None, cfg={'audio_config': {}})
assert not is_vlm(None, cfg={'hidden_size': 2048})
test_eq(read_config('/definitely/not/a/model'), {})

## The prompt cache

MLX exposes prompt-cache ownership directly. LiteRT retains KV state in its `Conversation`, and
llama.cpp re-renders the message list and reuses its longest matching KV prefix internally. Here
rishi tracks the rendered token ids itself, so turn two prefills only what turn one did not cover.

Keeping that correct means tracking which tokens the cache holds. Each turn renders the whole
conversation to token ids and compares it against `_cache_ids` with `common_prefix_len`:

- everything up to the first difference is already cached, and is skipped
- everything after it is stale, and is trimmed off the cache before generating

The stale part is a real case, not a theoretical one. A chat template does not necessarily re-render
an assistant turn as the exact tokens the model generated, and `recover_context` rewrites history
outright. Comparing tokens rather than trusting the cache makes both self-correcting, and a mismatch
costs only a re-prefill.

`trim_prompt_cache` removes tokens from the *end* of the cache, which is what is wanted here. It is
the same thing mlx-lm's own server does to reuse a cache across requests.


## MlxChat

The same callable surface as the other backends. The MLX-specific parts are `kv_bits`,
`draft_model`, `adapter_path`, `save_cache` and `load_cache`, and `chat.use.cached_tokens` from the
explicit prompt cache.


In [ ]:
#| export
class MlxChat(ToolLoopMixin, Chat):
    "Sync chat over a local MLX model: the `rishi.core.Chat` API over `ToolLoopMixin`'s tool loop."
    _runtime = 'mlx'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    _media_ok = False    # text-only; `MlxVlmChat` flips this
    _media_note = ("this MLX model is text-only. Install `pip install 'rishi[mlx-vlm]'` and use a vision "
                   "model (e.g. rishi.mlx.qwen3vl_4b) for image or audio input, or pass vlm=True.")
    mk_content, mk_msg, mk_msgs = staticmethod(mk_oai_content), staticmethod(mk_oai_msg), staticmethod(mk_oai_msgs)

    def __new__(cls, model=None, *, runtime=None, model_path=None, vlm=None, **kw):
        "Send a vision or audio repo to `MlxVlmChat`. `vlm=True` or `vlm=False` overrides the config sniff."
        if cls is MlxChat:
            tgt = model_path or core.split_runtime(model)[1] or ''
            if vlm if vlm is not None else (bool(tgt) and is_vlm(tgt)): return object.__new__(MlxVlmChat)
        return object.__new__(cls)

    @staticmethod
    def fmt2hist(msgs):
        "Normalize MLX messages to rishi history dicts."
        return mk_oai_msgs(msgs)
    @staticmethod
    def hist2fmt(msgs):
        "Canonical rishi history dicts -> OpenAI-style messages for the chat template (past-turn media -> placeholder)."
        return [to_oai_msg(strip_media(m)) for m in listify(msgs)]

    @classmethod
    def create_engine(cls, model_id=qwen3_4b, model_path=None, adapter_path=None, draft_model=None, **kw):
        'Load an MLX model + tokenizer, optionally with a LoRA adapter and a draft model. Override/`@patch` to customize.'
        src = str(model_path or model_id)
        model, tok, cfg = mlx_load(src, adapter_path=adapter_path, return_config=True, **kw)
        draft_id = str(draft_model) if isinstance(draft_model, (str, os.PathLike)) else None
        dm = mlx_load(draft_id)[0] if draft_id else draft_model
        return MlxEngine(model, tok, model_id=src, cfg=cfg, draft_model=dm, draft_model_id=draft_id)

    #: MLX spells the sampler's temperature `temp`, so only the window needs renaming
    _opt_map = {'ctx': 'ctx_limit'}
    _opt_skip = ('effort', 'tool_mode', 'api_key', 'base_url')   # nothing local takes these

    def __init__(self, model=None, *, runtime=None, model_path=None, opts=None,
                 vlm=None,               # force the vision class on or off, rather than sniffing
                 engine=None,            # an engine to share instead of loading one
                 adapter_path=None,      # a LoRA adapter to apply
                 draft_model=None,       # a smaller model, for speculative decoding
                 eng_kw=None,            # passed to `create_engine` verbatim
                 min_p=None,
                 prompt_cache=True, max_kv_size=None, kv_bits=None, kv_group_size=64,
                 quantized_kv_start=0,
                 tmpl_kw=None,           # passed to `apply_chat_template`
                 gen_kw=None,            # passed to `stream_generate` verbatim
                 **kw):                  # portable options; see `urai.ChatOpts`
        o = ChatOpts.create(opts, **kw)
        model = core.split_runtime(model)[1]
        model_id = None if model is None or is_path(model) else model
        model_path = model_path or (model if model and is_path(model) else None)
        if o.seed is not None: mx.random.seed(o.seed)
        self._own_engine = engine is None
        if engine is None:
            engine = self.create_engine(model_id or qwen3_4b, model_path, adapter_path=adapter_path,
                                        draft_model=draft_model, **(eng_kw or {}))
        self.engine = engine
        self._set_tools(o.tools)
        samp = {k: v for k, v in dict(temp=o.temp, top_k=o.top_k, top_p=o.top_p,
                                      min_p=min_p).items() if v is not None}
        self.sampler = make_sampler(**samp) if samp else None
        self.think = o.think
        self.max_output_tokens = o.max_output_tokens or DFLT_MAX_TOKENS
        store_attr('max_kv_size,kv_bits,kv_group_size,quantized_kv_start', self)
        self.tmpl_kw, self.gen_kw = tmpl_kw or {}, gen_kw or {}
        self.ctx_limit, self._ctx_tokens = o.ctx or engine.ctx_limit, 0
        self._cache, self._cache_ids = None, []
        if prompt_cache: self._reset_cache()
        self._setup(model, o)

    @property
    def tokenizer(self): return self.engine.tokenizer

    @property
    def token_count(self):
        "Tokens in the model context after the last turn (prompt + completion)."
        return self._ctx_tokens
    @property
    def cached_tokens(self):
        "How many tokens the KV cache currently holds."
        return len(self._cache_ids)

    def count_tokens(self, text):
        "Number of tokens in `text` per the model tokenizer."
        return len(self.engine.tokenize(text))

    def _reset_cache(self):
        "Start fresh main and draft KV caches. The next step re-prefills the whole conversation."
        self._cache = make_prompt_cache(self.engine.model, self.max_kv_size)
        if self.engine.draft_model is not None:
            self._cache += make_prompt_cache(self.engine.draft_model, self.max_kv_size)
        self._cache_ids = []
    def _recreate_conv(self):
        "History was rewritten by context recovery, so the cache no longer matches it. Drop it."
        if self._cache is not None: self._reset_cache()

    def _trim(self, n):
        "Drop the last `n` tokens from the KV cache. On a cache that cannot be trimmed, reset it and return False."
        if can_trim_prompt_cache(self._cache) and trim_prompt_cache(self._cache, n) == n:
            self._cache_ids = self._cache_ids[:len(self._cache_ids) - n]
            return True
        self._reset_cache()
        return False

    def _msgs(self):
        "OpenAI-style messages for the chat template: system prompt plus canonical history."
        return ([{'role': 'system', 'content': self.sp}] if self.sp else []) + self.hist2fmt(self.hist)

    def _prompt_ids(self):
        "Token ids for the whole conversation, rendered with the model's own chat template."
        kw = dict(add_generation_prompt=True)
        if self.toolspecs: kw['tools'] = self.toolspecs
        if self.think is not None: kw['enable_thinking'] = self.think
        return list(self.tokenizer.apply_chat_template(self._msgs(), **kw, **self.tmpl_kw))

    def _feed_ids(self):
        "`(all_ids, ids_still_to_prefill, n_cached)`, reusing however much of the prompt the KV cache covers."
        ids = self._prompt_ids()
        if self._cache is None: return ids, ids, 0
        n = common_prefix_len(ids, self._cache_ids)
        if (drop := len(self._cache_ids) - n) and not self._trim(drop): n = 0
        if n and n == len(ids):                 # nothing left to feed; give back one token to generate from
            n = n - 1 if self._trim(1) else 0
        return ids, ids[n:], n

    def _turn_sampler(self, turn):
        "This turn's sampler, when it overrode any sampling setting, else the chat's own."
        samp = {k: v for k, v in turn.items() if k in ('temp', 'top_k', 'top_p', 'min_p')}
        return make_sampler(**samp) if samp else self.sampler

    def _gen_kw(self, **turn):
        "Keyword arguments for `stream_generate`: sampler, prompt cache, KV quantization, draft model."
        t = self.map_opts(turn); t.pop('ctx_limit', None)   # a window is not a generation argument
        kw = dict(max_tokens=ifnone(t.pop('max_output_tokens', None), self.max_output_tokens))
        if (s := self._turn_sampler(t)) is not None: kw['sampler'] = s
        if self._cache is not None: kw['prompt_cache'] = self._cache
        if self.kv_bits: kw.update(kv_bits=self.kv_bits, kv_group_size=self.kv_group_size, quantized_kv_start=self.quantized_kv_start)
        if self.engine.draft_model is not None: kw['draft_model'] = self.engine.draft_model
        return {**kw, **self.gen_kw}

    def _generate(self, feed, **turn):
        "Yield mlx-lm `GenerationResponse` chunks for the token ids in `feed`."
        return stream_generate(self.engine.model, self.tokenizer, feed, **self._gen_kw(**turn))

    def _stream_step(self, **kw):
        "Stream one completion, yielding chunk dicts and leaving the merged `Resp` on `self._step_res`."
        ids, feed, cached = self._feed_ids()
        split, fin, gen, pt, gt = StreamSplit(), None, [], 0, 0
        try:
            for r in self._generate(feed, **kw):
                fin, pt, gt = r.finish_reason or fin, r.prompt_tokens, r.generation_tokens
                gen.append(r.token)
                yield from split.feed(r.text)
        finally:
            # `stream_generate` grows the cache a token at a time, so abandoning it half way leaves
            # `_cache_ids` short of what the cache holds and the next turn trims the wrong amount
            if self._cache is not None: self._cache_ids = ids + gen
        yield from split.finish()
        self._step_res = self._mk_resp(split, fin, cached, pt, gt)

    def _mk_resp(self, split, fin, cached, pt, gt):
        "Assemble one step's `Resp` from the parsed stream plus mlx-lm's token counts."
        res = {'role': 'assistant', 'content': split.text}
        if split.thought: res['channels'] = {'thought': split.thought}
        if split.tool_calls: res['tool_calls'] = split.tool_calls
        if fin == 'length': res['truncated'] = True
        res['usage'] = {'prompt_tokens': cached + pt, 'completion_tokens': gt,
                        'total_tokens': cached + pt + gt, 'cached_tokens': cached}
        return Resp(res)

    def _model_step(self, **kw):
        "One completion, normalized to a `Resp`. MLX generates by streaming, so this drains `_stream_step`."
        for _ in self._stream_step(**kw): pass
        return self._step_res

    def _cache_metadata(self):
        "Identity/settings that must match before a persisted KV cache can safely be reused."
        cfg = dict(model_id=self.engine.model_id, draft_model_id=self.engine.draft_model_id,
                   max_kv_size=self.max_kv_size, kv_bits=self.kv_bits,
                   kv_group_size=self.kv_group_size, quantized_kv_start=self.quantized_kv_start)
        return {'ids': json.dumps(self._cache_ids), 'rishi_mlx_cache': json.dumps(cfg, sort_keys=True)}

    def save_cache(self, fn):
        "Persist the KV cache and the tokens it holds, so a compatible chat can reuse it."
        if self._cache is None: raise ValueError('prompt_cache=False: there is no KV cache to save')
        save_prompt_cache(str(fn), self._cache, self._cache_metadata())
        return self
    def load_cache(self, fn):
        "Restore a compatible KV cache written by `save_cache`, rejecting model and setting mismatches."
        cache, md = load_prompt_cache(str(fn), return_metadata=True)
        got = md.get('rishi_mlx_cache')
        if not got: raise ValueError('not a rishi MLX cache (missing compatibility metadata)')
        expected = json.loads(self._cache_metadata()['rishi_mlx_cache'])
        if (actual := json.loads(got)) != expected:
            raise ValueError(f'incompatible MLX prompt cache: expected {expected}, got {actual}')
        self._cache, self._cache_ids = cache, json.loads(md.get('ids', '[]'))
        return self

    def _raw(self, msgs, max_output_tokens=None, think=None):
        "Stateless completion text for `msgs`, with no history and no prompt cache."
        tkw = {} if think is None else dict(enable_thinking=think)
        ids = list(self.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, **tkw))
        kw = dict(max_tokens=ifnone(max_output_tokens, self.max_output_tokens))
        if self.sampler is not None: kw['sampler'] = self.sampler
        return ''.join(r.text for r in stream_generate(self.engine.model, self.tokenizer, ids, **kw))

    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        """Stateless one-shot completion text, with any thinking stripped."""
        msgs = ([{'role': 'system', 'content': sp}] if sp else []) + [{'role': 'user', 'content': prompt}]
        return split_think(self._raw(msgs, max_output_tokens=max_tokens, think=think))[0]

    def _structured_call(self, prompt, schema, sp):
        "JSON reply for `schema`, parsed out of the model's text. mlx-lm has no grammar constraint built in, so this asks and then parses."
        p = f"{prompt}\n\nReply with only a JSON object matching this schema:\n{json.dumps(get_schema(schema)['input_schema'])}"
        txt = self._oneshot(p, sp, think=False)
        try: return json.loads(extract_fence(txt, 'json'))
        except (json.JSONDecodeError, TypeError): pass
        try: return json.loads(txt[txt.index('{'):txt.rindex('}') + 1])   # last resort: the outermost braces
        except Exception as e: raise ValueError(f"model did not return JSON; reply: {txt[:200]!r}") from e

    def close(self):
        "Release the model, if this Chat loaded it, and drop the cache. Idempotent."
        if getattr(self, '_own_engine', False) and getattr(self, 'engine', None) is not None:
            self.engine.close()
            self.engine = None
        self._cache, self._cache_ids = None, []

## Vision and audio

mlx-lm is text-only. Images and audio go through [mlx-vlm](https://github.com/Blaizzy/mlx-vlm), which
depends on mlx-lm and adds the vision and audio towers. Install it with `pip install 'rishi[mlx-vlm]'`.

`MlxVlmChat` is picked for you. `MlxChat(repo)` reads the repo's `config.json` and routes there when
it finds a vision or audio tower, so `Chat('mlx-community/Qwen3-VL-4B-Instruct-4bit')` just works.
Pass `vlm=True` or `vlm=False` to decide yourself.

It overrides only the three places mlx-vlm differs, which are loading, prompt building and the
generate call, and inherits history, callbacks, the tool loop and everything else unchanged. Two
honest limitations. mlx-vlm has no per-model tool-call parsers, so tool calls rely on `<tool_call>`
tags, which not every vision model emits. And it manages its own vision-feature cache rather than
mlx-lm's token KV cache, so cross-turn prefix reuse is off on this path.

Audio rides the same path. Hand a `Path` or `bytes` to a model with an audio tower and mlx-vlm
decodes it and resamples it to whatever rate the model's feature extractor wants. The one thing rishi
takes back from mlx-vlm is the thinking switch. mlx-vlm asks the chat template for
`enable_thinking=False`, which prefills an empty `<think></think>` block, and a model handed a
finished thought can simply end the turn. Qwen3-Omni transcribes nothing at all. rishi passes its own
`think` through instead, so the default is whatever the model's template does on its own.


In [ ]:
#| export
def _media_bytes(p):
    "Raw bytes out of an OpenAI-style image/audio content part."
    if p.get('type') == 'image_url':
        url = p['image_url']['url'] if isinstance(p['image_url'], dict) else p['image_url']
        if not url.startswith('data:'):
            raise ValueError('MLX VLM accepts local image bytes/Paths, not remote image URLs')
        return b64decode(url.split(',', 1)[1])
    return b64decode(p['input_audio']['data'])

class MlxVlmChat(MlxChat):
    "MLX chat for vision and audio models, via `mlx-vlm`. Same API as `MlxChat`, and media in the live turn reaches the model."
    _media_ok = True

    @classmethod
    def create_engine(cls, model_id=qwen3vl_4b, model_path=None, adapter_path=None, draft_model=None, **kw):
        'Load a vision or audio model with `mlx_vlm.load`. The "tokenizer" here is a HF processor.'
        try: from mlx_vlm import load as vlm_load
        except ImportError as e:
            raise ImportError(f"vision/audio MLX models need mlx-vlm ({e}). "
                              "Install it with: pip install 'rishi[mlx-vlm]'") from None
        src = str(model_path or model_id)
        if adapter_path: kw['adapter_path'] = adapter_path
        model, proc = vlm_load(src, **kw)
        return MlxEngine(model, proc, model_id=src, cfg=read_config(src))

    def _reset_cache(self):
        "No token-level KV cache on this path: mlx-vlm keeps its own vision-feature cache instead."
        self._cache, self._cache_ids = None, []

    def _tmpdir(self):
        "A per-chat temp directory to stage media files for mlx-vlm."
        if getattr(self, '_td', None) is None: self._td = TemporaryDirectory()
        return Path(self._td.name)

    def _stage(self, data, ext):
        "Write `data` to a temp file and return its path (mlx-vlm takes paths/PIL images, not bytes)."
        p = self._tmpdir()/f'{len(list(self._tmpdir().iterdir()))}{ext}'
        p.write_bytes(data)
        return str(p)

    def _turn_media(self):
        "Image and audio paths from the live turn, staged for mlx-vlm."
        imgs, auds = [], []
        for p in ((self.turn_msg or {}).get('content') or []):
            if not (isinstance(p, dict) and is_media(p)): continue
            if not (data := _media_bytes(p)): continue
            if p['type'] == 'image_url': imgs.append(self._stage(data, '.png'))
            else: auds.append(self._stage(data, f".{p['input_audio'].get('format', 'wav')}"))
        return imgs, auds

    def _prompt(self, imgs, auds):
        "The formatted prompt string mlx-vlm expects, including media counts and tool schemas."
        from mlx_vlm.prompt_utils import apply_chat_template as vlm_tmpl
        # mlx-vlm defaults to `enable_thinking=False`, which prefills an empty `<think></think>` block
        # into the assistant turn. A model told it has already finished thinking can end the turn on the
        # spot - Qwen3-Omni answers an audio prompt with nothing at all. Pass rishi's own `think` through
        # instead, so `think=None` (the default) leaves the model's chat template default alone.
        kw = dict(num_images=len(imgs), num_audios=len(auds), enable_thinking=self.think)
        if self.toolspecs: kw['tools'] = self.toolspecs
        return vlm_tmpl(self.tokenizer, self.engine.cfg, self._msgs(), **{**kw, **self.tmpl_kw})

    def _stream_step(self, **turn):
        "Stream one completion through mlx-vlm, leaving the merged `Resp` on `self._step_res`."
        from mlx_vlm import stream_generate as vlm_stream
        imgs, auds = self._turn_media()
        t = self.map_opts(turn); t.pop('ctx_limit', None)
        kw = dict(max_tokens=ifnone(t.pop('max_output_tokens', None), self.max_output_tokens),
                  **self.gen_kw)
        if imgs: kw['image'] = imgs
        # mlx-vlm decodes each audio file and resamples it to the processor's own feature-extractor
        # rate, so the source rate is its business, not ours - handing it one is how you get silence
        if auds: kw['audio'] = auds
        split, fin, pt, gt = StreamSplit(), None, 0, 0
        if (s := self._turn_sampler(t)) is not None: kw['sampler'] = s
        for r in vlm_stream(self.engine.model, self.tokenizer, self._prompt(imgs, auds), **kw):
            fin = getattr(r, 'finish_reason', None) or fin
            pt, gt = getattr(r, 'prompt_tokens', pt), getattr(r, 'generation_tokens', gt)
            yield from split.feed(r.text)
        yield from split.finish()
        self._step_res = self._mk_resp(split, fin, 0, pt, gt)

    def close(self):
        "Release the model and clean up any staged media files."
        if getattr(self, '_td', None) is not None: self._td.cleanup(); self._td = None
        super().close()

### Tests

Fake-tokenizer tests run everywhere. The `#| eval: false` cells need Apple Silicon and downloaded
weights.


In [ ]:
#| hide
from types import SimpleNamespace as NS

class _FakeTok:
    "Tokenizes by word, and renders a conversation as `role:text` tokens - enough to exercise prefix reuse."
    def encode(self, s): return [hash(w) % 10000 for w in s.split()]
    def apply_chat_template(self, msgs, add_generation_prompt=False, tools=None, **kw):
        s = ' '.join(f"{m['role']}:{m.get('content') or ''}" for m in msgs)
        if tools: s = f'tools:{len(tools)} ' + s
        return self.encode(s)

def _chunk(text, token, fin=None, pt=5, gt=1):
    return NS(text=text, token=token, finish_reason=fin, prompt_tokens=pt, generation_tokens=gt,
              from_draft=False, logprobs=None, prompt_tps=0., generation_tps=0., peak_memory=0.)

class _FakeMlxChat(MlxChat):
    "MlxChat with mlx-lm swapped out: `_generate` replays a script, and the KV cache is a token list."
    def __init__(self, script=None, **kw):
        self.script, self.fed = list(script or []), []
        eng = MlxEngine(object(), _FakeTok(), model_id='fake', cfg={'max_position_embeddings': 4096})
        super().__init__(engine=eng, prompt_cache=False, **kw)
        self._cache, self._cache_ids = [], []          # a plain list stands in for the KV cache
    def _reset_cache(self): self._cache, self._cache_ids = [], []
    def _trim(self, n):
        self._cache_ids = self._cache_ids[:len(self._cache_ids) - n]
        return True
    def _generate(self, feed, max_output_tokens=None):
        self.fed.append(list(feed))
        return iter(self.script.pop(0) if self.script else [_chunk('ok', 1, 'stop')])

In [ ]:
#| hide
# speculative decoding needs one KV cache per layer of *both* models, so a real draft model doubles
# the cache. `make_prompt_cache` walks the actual layers, so this only means anything on real weights.
eng = MlxChat.create_engine(qwen3_06b, draft_model=qwen3_06b)
spec = MlxChat(engine=eng)
test_eq(len(spec._cache), 2 * len(eng.model.layers))
test_eq(spec.cached_tokens, 0)                  # nothing prefilled yet
spec.close(); eng.close()

# mlx-vlm stages media as files, so a remote image URL has nothing to stage and says so
test_fail(lambda: _media_bytes({'type': 'image_url', 'image_url': {'url': 'https://example.com/x.png'}}),
          contains='not remote image URLs')

# audio parts decode back to the exact bytes that were handed in
wav = Path('speech.wav').read_bytes()
test_eq(_media_bytes(mk_oai_content(Path('speech.wav'))), wav)


In [ ]:
#| hide
# turn 1 prefills everything; turn 2 only feeds what the cache doesn't already cover
c = _FakeMlxChat([[_chunk('hello', 11, 'stop')], [_chunk('again', 12, 'stop')]])
c('hi there')
first_feed = c.fed[0]
test_eq(c.cached_tokens, len(first_feed) + 1)          # prompt + the generated token
c('more please')
assert len(c.fed[1]) < len(first_feed) + 4             # only the new tail was prefilled
test_eq(c.use.cached_tokens > 0, True)                 # and it was reported as cached

# a rewritten history invalidates the tail: the cache is trimmed back to the common prefix
c = _FakeMlxChat([[_chunk('a', 1, 'stop')], [_chunk('b', 2, 'stop')]])
c('first question')
c.hist[0] = {'role': 'user', 'content': 'totally different question'}
c('second')
assert c.fed[1] != c.fed[0]

# nothing new to feed (the whole prompt is already cached) still leaves one token to generate from
c = _FakeMlxChat(messages=['hello there', {'role': 'assistant', 'content': 'hi back'}])
ids = c._prompt_ids()
assert ids                                              # sanity: this fake really does tokenize
c._cache_ids = list(ids)
all_ids, feed, cached = c._feed_ids()
test_eq(len(feed), 1)
test_eq(cached, len(ids) - 1)

# an empty prompt is degenerate, but must not drive the cache offset negative
c = _FakeMlxChat()
test_eq(c._feed_ids(), ([], [], 0))

# an untrimmable cache falls back to a full re-prefill rather than corrupting state
class _NoTrim(_FakeMlxChat):
    def _trim(self, n): self._reset_cache(); return False
c = _NoTrim()
c._cache_ids = [999, 998, 997]                          # nothing in common with the real prompt
ids, feed, cached = c._feed_ids()
test_eq((cached, feed), (0, ids))

In [ ]:
#| hide
# thinking, tool calls and truncation all come out of the streamed text
c = _FakeMlxChat([[_chunk('<think>hmm</think>', 1), _chunk('the answer is 4', 2, 'stop')]])
r = c('2+2?')
test_eq(resp_text(r), 'the answer is 4')
test_eq(thought(r), 'hmm')
test_eq(r['usage']['completion_tokens'], 1)

c = _FakeMlxChat([[_chunk('<tool_call>{"name": "add", "arguments": {"a": 1, "b": 2}}</tool_call>', 1, 'stop')],
                  [_chunk('that is 3', 2, 'stop')]])
def add(
    a: int,  # first addend
    b: int,  # second addend
) -> int:
    'Add two integers.'
    return a + b
c = _FakeMlxChat([[_chunk('<tool_call>{"name": "add", "arguments": {"a": 1, "b": 2}}</tool_call>', 1, 'stop')],
                  [_chunk('that is 3', 2, 'stop')]], tools=[add])
test_eq(resp_text(c('add 1 and 2')), 'that is 3')
test_eq([m['content'] for m in c.hist if m['role'] == 'tool'], ['3'])       # the tool really ran

# truncation is flagged from mlx-lm's finish_reason
c = _FakeMlxChat([[_chunk('cut off here', 1, 'length')]])
assert truncated(c('go'))

# tools reach the chat template
c = _FakeMlxChat(tools=[add])
assert c._prompt_ids() != _FakeMlxChat()._prompt_ids()

# a text-only model refuses media rather than silently dropping it
c = _FakeMlxChat()
png = b'\x89PNG\r\n\x1a\n' + b'0' * 8
test_fail(lambda: c([png, 'what is this?']), contains='text-only')

In [ ]:
#| hide
import rishi.mlx
# routing: an mlx repo id resolves to this backend without an explicit runtime=
test_eq(core.resolve_runtime('mlx-community/Qwen3-4B-4bit'), ('mlx', 'mlx-community/Qwen3-4B-4bit'))
test_eq(core.resolve_runtime('mlx/Qwen3-4B'), ('mlx', 'Qwen3-4B'))
test_eq(core.get_runtime('mlx'), rishi.mlx.MlxChat)

# vlm=True/False overrides the config sniff, and __new__ picks the class without loading anything
test_eq(type(rishi.mlx.MlxChat.__new__(rishi.mlx.MlxChat, 'anything', vlm=True)), rishi.mlx.MlxVlmChat)
test_eq(type(rishi.mlx.MlxChat.__new__(rishi.mlx.MlxChat, 'anything', vlm=False)), rishi.mlx.MlxChat)
assert isinstance(rishi.mlx.MlxVlmChat.__new__(rishi.mlx.MlxVlmChat), rishi.mlx.MlxVlmChat)

# history is portable: a litert-shaped conversation loads straight into an mlx chat
h = [{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'hello'}]
test_eq(rishi.mlx.MlxChat.fmt2hist(h), h)
test_eq(rishi.mlx.MlxChat.hist2fmt(h), h)

# and the whole chain, core.Chat -> MlxChat -> MlxVlmChat, without loading a model
test_eq(type(Chat.__new__(Chat, 'mlx/some-vision-model', vlm=True)), rishi.mlx.MlxVlmChat)
test_eq(type(Chat.__new__(Chat, 'mlx-community/Qwen3-4B-4bit', vlm=False)), rishi.mlx.MlxChat)

## Examples (manual)

Use `qwen3_4b`, about 2.5GB, unless you need vision or audio. The demos below cover cache reuse,
thinking, the MLX-only knobs, and VLM routing.


In [ ]:
#| eval: false
chat = MlxChat(qwen3_06b, sp='You are concise.', think=False)

In [ ]:
#| eval: false
r = chat('Reply with exactly: pong')
assert 'pong' in resp_text(r).lower(), resp_text(r)
r

pong

In [ ]:
#| eval: false
# multi-turn: the second turn must actually remember the first
chat('My favourite number is 17. Remember it.')
r = chat('What is my favourite number? Reply with just the number.')
assert '17' in resp_text(r), resp_text(r)
print(chat.use)

total=75|in=72|out=3|turns=1|cached=44


In [ ]:
#| eval: false
# Real MLX KV-cache test: capture the exact token tail handed to mlx-lm on each turn.
import time
chat2 = MlxChat(qwen3_06b, sp='You are concise.', think=False, max_output_tokens=16)
fed, orig_generate = [], chat2._generate
def counted_generate(ids, max_output_tokens=None):
    fed.append(len(ids))
    return orig_generate(ids, max_output_tokens)
chat2._generate = counted_generate
t0 = time.time(); chat2(('cache verification context ' * 96) + '\nReply with exactly: stored'); t1 = time.time() - t0
u1, first_feed = chat2.use, fed[-1]
t0 = time.time(); r = chat2('What exact word did I ask you to reply with? Reply with only that word.'); t2 = time.time() - t0
u2, second_feed = chat2.use, fed[-1]
print(dict(turn1_seconds=round(t1, 2), turn2_seconds=round(t2, 2), first_feed=first_feed,
           second_feed=second_feed, cached=u2.cached_tokens, answer=resp_text(r)))
assert u1.cached_tokens == 0, 'first turn has nothing to reuse'
assert u2.cached_tokens > 0, 'second turn should have reused the cache'
assert second_feed < first_feed, 'turn two re-prefilled the full old prompt'
assert chat2.cached_tokens > 0
assert 'stored' in resp_text(r).lower(), resp_text(r)
chat2.close()


{'turn1_seconds': 0.25, 'turn2_seconds': 0.08, 'first_feed': 315, 'second_feed': 32, 'cached': 311, 'answer': 'stored'}


In [ ]:
#| eval: false
# and with the cache off, nothing is ever reported as cached
plain = MlxChat(qwen3_4b, prompt_cache=False, think=False)
plain('Say hi.'); plain('Say hi again.')
test_eq(plain.use.cached_tokens, 0)
plain.close()

In [ ]:
#| eval: false
# streaming, with thinking rendered as a blockquote
thinker = MlxChat(qwen3_4b, think=True, max_output_tokens=512)
md = display_stream(thinker('Briefly: why is the sky blue?', stream=True))
assert thought(thinker.turn_res), 'expected a <think> block from a thinking model'
thinker.close()

> **🧠 Thinking**
>
> 
> Okay, the user is asking why the sky is blue. I need to explain this in a brief way. Let me recall the scientific explanation. The sky appears blue because of Rayleigh scattering. The atmosphere scatters sunlight, and shorter wavelengths (blue) are scattered more than longer ones (red). So when we look up, the blue light is scattered in all directions, making the sky appear blue. But wait, I should make sure I get the details right. Also, mention that the sun's light is white, but the scattering makes it look blue. Maybe add that the reason is the way the atmosphere interacts with sunlight. Keep it simple and concise. Avoid jargon. Maybe start with the main point, then the reason, then the effect. That should cover it.
> 

The sky appears blue because sunlight scatters more in the atmosphere when it interacts with the gases and particles in the air. Shorter wavelengths (blue light) are scattered more than longer wavelengths (red light), so when we look up, the scattered blue light dominates, giving the sky its blue color.

In [ ]:
#| eval: false
# quantized KV cache: same answer, less memory for long contexts
q = MlxChat(qwen3_4b, kv_bits=4, quantized_kv_start=256, think=False)
print(resp_text(q('Name three primes.')))
q.close()

Sure! Here are three prime numbers: **2, 3, and 5**. 

A prime number is a number greater than 1 that has no positive divisors other than 1 and itself.


In [ ]:
#| eval: false
# speculative decoding: a small draft model proposes, the big one verifies
import time
spec = MlxChat(qwen3_8b, draft_model=qwen3_06b, think=False, max_output_tokens=256)
t0 = time.time(); spec('Write two sentences about monsoons.'); print(f'{time.time()-t0:.1f}s', spec.use)
spec.close()

1.9s total=60|in=20|out=40|turns=1


In [ ]:
#| eval: false
# a warmed cache can be saved and reloaded, so a long system prompt is prefilled once, ever
import tempfile
warm = MlxChat(qwen3_4b, sp='You are a terse assistant.', think=False)
warm('Remember the codeword: albatross.')
fn = Path(tempfile.mkdtemp())/'warm.safetensors'
warm.save_cache(fn)
n = warm.cached_tokens
warm.close()

again = MlxChat(qwen3_4b, sp='You are a terse assistant.', think=False)
again.load_cache(fn)
test_eq(again.cached_tokens, n)
again.close()

In [ ]:
#| eval: false
# the point of a shared history format: carry a conversation from one backend to another
from rishi.llama import LlamaChat, qwen3_17b as llama_qwen

lc = LlamaChat(llama_qwen, n_ctx=4096)
lc('My name is Karthik. Remember it.')
hist = lc.hist                      # canonical rishi history, backend-independent
lc.close()

mc = MlxChat(qwen3_4b, messages=hist, think=False)
r = mc('What is my name? Reply with just the name.')
assert 'karthik' in resp_text(r).lower(), resp_text(r)
mc.close()
resp_text(r)

llama_context: n_ctx_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


'Karthik'

### Images and audio

These need `pip install 'rishi[mlx-vlm]'` and a vision or audio model. The class is selected for you.


In [ ]:
#| eval: false
vchat = MlxChat(qwen3vl_4b, max_output_tokens=256)
test_eq(type(vchat), MlxVlmChat)              # routed automatically from the repo's config.json

img = Path('images.jpeg').read_bytes()
r = vchat([img, 'What is in this image? One sentence.'])
print(resp_text(r))
assert resp_text(r).strip()

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


A happy German Shepherd dog with its tongue out, standing on a gravel path outdoors.


In [ ]:
#| eval: false
# an omni build (audio tower) transcribes; gemma-4-e4b does vision *and* audio in about 5GB
achat = MlxChat(gemma4_e4b, max_output_tokens=256)
r = achat([Path('speech.wav'), 'Transcribe this audio.'])
print(resp_text(r))
assert 'masquerade' in resp_text(r).lower()

# mlx-vlm's own default is `enable_thinking=False`, which prefills an empty `<think></think>` block
# into the reply. rishi passes `think` through instead: with the block, `qwen3omni_30b` ends the turn
# without transcribing anything at all.
assert '</think>' not in achat._prompt([], ['speech.wav'])
achat.close()

/Users/71293/code/personal/orgs/ramabana/.venv/lib/python3.13/site-packages/transformers/audio_utils.py:723: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(


Dancing in the masquerade, idol truth in plain sight jaded. Pop, roll, click, shot. Buhulabi today or not? But such a tide as moving seems asleep, too full for sound and foam. When that which drew from out the boundless deep turns again home, twilight and evening bell, and after that.


In [ ]:
#| eval: false
for c in (chat, vchat):
    try: c.close()
    except Exception: pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Sharing one engine

`MlxBroker` keeps one loaded MLX model in a long-lived process and serves an `MlxChat` per client
over a Unix socket, with one set of weights and a prompt cache each. See [Sharing one
engine](core.html#sharing-one-engine).


In [ ]:
#| export
class MlxBroker(ChatBroker):
    'Shared MLX model broker. Each client gets an isolated `MlxChat` prompt cache.'
    def __init__(self, address, engine=None, model_id=qwen3_4b, **engine_kw):
        super().__init__(address, MlxChat, engine, model_id, **engine_kw)

In [ ]:
#| hide
test_eq(MlxBroker('/tmp/rishi-mlx.sock').chat_cls, MlxChat)   # protocol and lifetime are `ChatBroker`'s, tested there